# บทที่ 8: Hashing และ Hash Table> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)---

## 8.1 แนวคิดHash table แปลง **key** เป็น **index** ของอาร์เรย์ด้วย *hash function* → เข้าถึงได้เฉลี่ย \(O(1)\)**คุณสมบัติของ hash function ที่ดี**1. คำนวณเร็ว2. กระจายค่าสม่ำเสมอ (uniform distribution)3. key เดิมให้ค่าเดิมเสมอ (deterministic)**Collision** = key ต่างกันได้ index เดียวกัน → แก้ได้ 2 แนวทางหลัก- **Separate chaining:** แต่ละช่องเก็บลิสต์ของคู่ (key, value)- **Open addressing:** หาช่องว่างถัดไป (linear probing, quadratic probing, double hashing)**Load factor** \(\alpha = n / m\) (จำนวนข้อมูล / จำนวนช่อง) — เมื่อ \(\alpha\) สูงต้อง **rehash** (ขยายตารางแล้วคำนวณ index ใหม่ทั้งหมด)

In [ ]:
class HashTableChaining:    # Separate chaining: buckets[i] เป็นลิสต์ของคู่ [key, value]    def __init__(self, capacity=8):        self.cap = capacity        self.n = 0        self.buckets = [[] for _ in range(self.cap)]    def _index(self, key):        return hash(key) % self.cap    def put(self, key, value):        b = self.buckets[self._index(key)]        for pair in b:            if pair[0] == key:                pair[1] = value                 # อัปเดตค่าเดิม                return        b.append([key, value])        self.n += 1        if self.n / self.cap > 0.75:            # load factor เกินเกณฑ์ -> ขยาย            self._rehash()    def get(self, key, default=None):        for k, v in self.buckets[self._index(key)]:            if k == key: return v        return default    def remove(self, key):        b = self.buckets[self._index(key)]        for i, (k, _) in enumerate(b):            if k == key:                b.pop(i); self.n -= 1                return True        return False    def _rehash(self):        old = self.buckets        self.cap *= 2        self.buckets = [[] for _ in range(self.cap)]        self.n = 0        for b in old:            for k, v in b:                self.put(k, v)                  # O(n) แต่เกิดไม่บ่อย -> amortized O(1)    def stats(self):        lens = [len(b) for b in self.buckets]        return f"cap={self.cap} n={self.n} load={self.n/self.cap:.2f} bucket ยาวสุด={max(lens)}"ht = HashTableChaining(4)for i, name in enumerate(["ana", "ben", "cara", "dan", "eve"]):    ht.put(name, i)print(ht.stats())print("get('cara') =", ht.get("cara"), "| remove('ben') =", ht.remove("ben"))print("get('ben')  =", ht.get("ben", "ไม่พบ"))

In [ ]:
class HashTableLinearProbing:    # Open addressing: เก็บในอาร์เรย์เดียว ใช้ tombstone ('DELETED') ตอนลบ    _EMPTY, _DELETED = object(), object()    def __init__(self, capacity=8):        self.cap, self.n = capacity, 0        self.keys = [self._EMPTY] * self.cap        self.vals = [None] * self.cap    def _probe(self, key):        i = hash(key) % self.cap        first_deleted = -1        for step in range(self.cap):            j = (i + step) % self.cap           # linear probing            k = self.keys[j]            if k is self._EMPTY:                return (first_deleted if first_deleted >= 0 else j), False            if k is self._DELETED:                if first_deleted < 0: first_deleted = j            elif k == key:                return j, True        raise RuntimeError('table full')    def put(self, key, value):        j, found = self._probe(key)        self.keys[j], self.vals[j] = key, value        if not found:            self.n += 1            if self.n / self.cap > 0.6:                self._resize()    def get(self, key, default=None):        j, found = self._probe(key)        return self.vals[j] if found else default    def remove(self, key):        j, found = self._probe(key)        if found:            self.keys[j], self.vals[j] = self._DELETED, None            self.n -= 1        return found    def _resize(self):        items = [(k, v) for k, v in zip(self.keys, self.vals)                 if k is not self._EMPTY and k is not self._DELETED]        self.cap *= 2        self.keys = [self._EMPTY] * self.cap        self.vals = [None] * self.cap        self.n = 0        for k, v in items:            self.put(k, v)lp = HashTableLinearProbing(8)for w in ["python", "java", "c", "go", "rust"]:    lp.put(w, len(w))print("get('rust') =", lp.get("rust"), "| cap =", lp.cap, "| n =", lp.n)lp.remove("java")print("หลังลบ java -> get('java') =", lp.get("java", "ไม่พบ"), "| get('go') =", lp.get("go"))

In [ ]:
# เปรียบเทียบ hash function: อันที่ไม่ดีทำให้เกิด collision กระจุกตัวdef bad_hash(s, m):    return sum(ord(c) for c in s) % m        # anagram ชนกันหมด เช่น 'abc' กับ 'cba'def poly_hash(s, m, base=31):    h = 0    for c in s:        h = (h * base + ord(c)) % m          # polynomial rolling hash: คำนึงถึงตำแหน่ง    return hwords = ["listen", "silent", "enlist", "tinsel", "data", "structure", "algorithm"]m = 8for f in (bad_hash, poly_hash):    buckets = {}    for w in words:        buckets.setdefault(f(w, m), []).append(w)    worst = max(len(v) for v in buckets.values())    print(f"{f.__name__:>10}: ใช้ {len(buckets)}/{m} ช่อง, bucket ยาวสุด = {worst}")

In [ ]:
# การประยุกต์ใช้ hashing ในโจทย์จริงfrom collections import Counter, defaultdict# 1) หาคำที่ปรากฏบ่อยสุด -> O(n)def most_common_words(text, k=3):    return Counter(text.lower().split()).most_common(k)# 2) จัดกลุ่ม anagram โดยใช้ตัวอักษรที่เรียงแล้วเป็น key -> O(n * L log L)def group_anagrams(words):    g = defaultdict(list)    for w in words:        g[''.join(sorted(w))].append(w)    return list(g.values())# 3) substring ที่ไม่มีอักขระซ้ำ ยาวสุด (sliding window + hash) -> O(n)def longest_unique_substring(s):    last, start, best = {}, 0, (0, '')    for i, c in enumerate(s):        if c in last and last[c] >= start:            start = last[c] + 1              # ย้ายขอบซ้ายข้ามตัวซ้ำ        last[c] = i        if i - start + 1 > best[0]:            best = (i - start + 1, s[start:i+1])    return bestprint(most_common_words("data structure and data algorithm and data"))print(group_anagrams(["listen", "silent", "enlist", "google", "elgoog"]))print(longest_unique_substring("abcabcbb"))

---## แบบฝึกหัดบทที่ 81. ตรวจว่าอาร์เรย์มีสมาชิกซ้ำหรือไม่ ด้วย \(O(n)\)2. หาความยาวของ subarray ที่ผลรวมเท่ากับ k (ใช้ prefix sum + hash map)3. เขียน `LRUCache` ด้วย hash map + doubly linked list ให้ทุก operation เป็น \(O(1)\)4. หาจุดตัด (intersection) และหาว่าสองสตริงเป็น anagram กันหรือไม่ ด้วย \(O(n)\)

### เฉลยบทที่ 8

In [ ]:
from collections import Counter, OrderedDict# ข้อ 1def has_duplicate(a):    return len(set(a)) != len(a)# ข้อ 2: ถ้า prefix[j] - prefix[i] = k -> ช่วง (i, j] มีผลรวม kdef longest_subarray_sum_k(a, k):    first_index, s, best = {0: -1}, 0, 0    for i, x in enumerate(a):        s += x        if s - k in first_index:            best = max(best, i - first_index[s - k])        first_index.setdefault(s, i)      # เก็บตำแหน่งแรกสุดเท่านั้น เพื่อให้ช่วงยาวสุด    return best# ข้อ 3: ใช้ OrderedDict (แนวคิดเดียวกับ hash map + doubly linked list)class LRUCache:    def __init__(self, capacity):        self.cap = capacity        self.d = OrderedDict()    def get(self, key):        if key not in self.d: return -1        self.d.move_to_end(key)                     # ทำให้เป็นตัวที่ใช้ล่าสุด        return self.d[key]    def put(self, key, value):        if key in self.d: self.d.move_to_end(key)        self.d[key] = value        if len(self.d) > self.cap:            self.d.popitem(last=False)              # ตัดตัวที่ใช้นานสุดออก# ข้อ 4def intersection(a, b):    return list(set(a) & set(b))def is_anagram(s, t):    return Counter(s) == Counter(t)print(has_duplicate([1,2,3,1]))print(longest_subarray_sum_k([10, 5, 2, 7, 1, 9], 15))c = LRUCache(2); c.put('a',1); c.put('b',2); c.get('a'); c.put('c',3)print("LRU เหลือ:", list(c.d.items()))            # 'b' ถูกตัดออกprint(intersection([1,2,3],[2,3,4]), is_anagram("listen","silent"))